# Week 2 — DINOv2 Vision Embeddings (CIFAR-10 binary)
**Project**: Quantum Kernels — Vision Encoder Comparison

**Purpose:** the sharpest vision-encoder question. DINOv2 is self-supervised —
trained on images alone, NO language. Comparing it to CLIP (vision-language
contrastive) on CIFAR-10 tests whether the quantum-vs-classical finding depends
on the encoder being language-trained at all.

**DINOv2 differences from CLIP image encoder:**
- Self-supervised (no text), uses [CLS] token of last_hidden_state
- 768-dim output (vs CLIP's 512)
- AutoImageProcessor + AutoModel interface

**Task:** airplane vs automobile (same as your CLIP CIFAR-10)
**Metric:** binary F1 | **Runtime:** ~6 min

In [ ]:
!pip install once: pip install datasets transformers torch torchvision Pillow scikit-learn numpy --quiet
import numpy as np, time, os, pickle, shutil, json
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')
import torch
from transformers import AutoImageProcessor, AutoModel
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, f1_score
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {DEVICE}")

---
## Step 0: Local storage

In [ ]:
USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_DIR = '/content/drive/MyDrive/qml_project'
else:
    SAVE_DIR = 'qml_project'
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"Save directory: {SAVE_DIR}")
def _persist_and_download(path):
    if os.path.abspath(SAVE_DIR) != os.getcwd():
        shutil.copy(path, os.path.join(SAVE_DIR, os.path.basename(path)))
        print(f"  saved -> {SAVE_DIR}/{os.path.basename(path)}")
    try:
        from google.colab import files
        files.download(path)
        print(f"  download -> {os.path.basename(path)}")
    except Exception as e:
        print(f"  (download skipped: {type(e).__name__} — saved above)")
def checkpoint(filename, save_fn):
    save_fn(filename); _persist_and_download(filename)
def save_figure(fig, name):
    for ext in ('png','pdf'):
        p = f"{name}.{ext}"; fig.savefig(p, bbox_inches='tight', dpi=150); _persist_and_download(p)
def _find(fname):
    dp = os.path.join(SAVE_DIR, fname); return dp if os.path.exists(dp) else fname
print("Helpers ready (Drive + download).")

---
## Step 1: Load CIFAR-10 (airplane vs automobile)

In [ ]:
CIFAR_CLASSES={0:'airplane',1:'automobile'}
import random; random.seed(42)
try:
    from datasets import load_dataset
    print("Loading uoft-cs/cifar10...")
    ds = load_dataset("uoft-cs/cifar10")
    img_col = 'img' if 'img' in ds['train'].column_names else 'image'
    def collect(split, n_per):
        imgs,lbls=[],[]; c={0:0,1:0}
        for s in ds[split]:
            if s['label'] in [0,1] and c[s['label']]<n_per:
                imgs.append(s[img_col]); lbls.append(s['label']); c[s['label']]+=1
            if all(v>=n_per for v in c.values()): break
        return imgs,lbls
    train_images,train_labels_raw = collect('train',600)
    test_images,test_labels_raw   = collect('test',130)
except Exception as e:
    print(f"HF failed ({e}); using torchvision...")
    import torchvision
    def collect_tv(train,n_per):
        d=torchvision.datasets.CIFAR10('./data',train=train,download=True)
        imgs,lbls=[],[]; c={0:0,1:0}
        for img,lbl in d:
            if lbl in [0,1] and c[lbl]<n_per: imgs.append(img); lbls.append(lbl); c[lbl]+=1
            if all(v>=n_per for v in c.values()): break
        return imgs,lbls
    train_images,train_labels_raw=collect_tv(True,600)
    test_images,test_labels_raw=collect_tv(False,130)

def balance(imgs,lbls,n_per):
    c0=[(i,l) for i,l in zip(imgs,lbls) if l==0][:n_per]
    c1=[(i,l) for i,l in zip(imgs,lbls) if l==1][:n_per]
    comb=c0+c1; random.shuffle(comb)
    return [x[0] for x in comb], np.array([x[1] for x in comb])
train_images,train_labels=balance(train_images,train_labels_raw,550)
test_images,test_labels=balance(test_images,test_labels_raw,110)
print(f"Train {len(train_images)} | Test {len(test_images)}")

---
## Step 2: Extract DINOv2 embeddings ([CLS] token) — checkpoint after

In [ ]:
MODEL_NAME="facebook/dinov2-base"
processor=AutoImageProcessor.from_pretrained(MODEL_NAME)
model=AutoModel.from_pretrained(MODEL_NAME).to(DEVICE); model.eval()
print(f"DINOv2 loaded. Output dim: 768 (CLS token)")

def extract(images, batch_size=32):
    embs=[]
    for i in range(0,len(images),batch_size):
        batch=[im.convert('RGB') for im in images[i:i+batch_size]]
        inp=processor(images=batch, return_tensors='pt').to(DEVICE)
        with torch.no_grad():
            out=model(**inp)
            cls=out.last_hidden_state[:,0,:]    # CLS token = image representation
            cls=cls/torch.linalg.norm(cls,dim=-1,keepdim=True)
        embs.append(cls.cpu().numpy())
    return np.vstack(embs)

t0=time.time()
train_emb=extract(train_images); test_emb=extract(test_images)
print(f"Done {time.time()-t0:.0f}s | train {train_emb.shape} test {test_emb.shape}")
assert train_emb.shape[1]==768
checkpoint('cifar10_dinov2_raw.npz', lambda p: np.savez(p, train_emb=train_emb, test_emb=test_emb,
                                                        train_labels=train_labels, test_labels=test_labels))

---
## Step 3: PCA + normalize

In [ ]:
pca_objects={n:PCA(n_components=n,random_state=42).fit(train_emb) for n in [4,8,12,16]}
print("PCA variance (768-dim DINOv2 -> reduced):")
for n in [4,8,12,16]: print(f"  {n:>2} -> {np.sum(pca_objects[n].explained_variance_ratio_)*100:.1f}%")
pca_8=pca_objects[8]
X_train_scaled=MinMaxScaler((0,np.pi)).fit_transform(pca_8.transform(train_emb))
scaler=MinMaxScaler(feature_range=(0,np.pi)); scaler.fit(pca_8.transform(train_emb))
X_train_scaled=scaler.transform(pca_8.transform(train_emb))
X_test_scaled=np.clip(scaler.transform(pca_8.transform(test_emb)),0,np.pi)
checkpoint('cifar10_dinov2_pca_scaler.pkl', lambda p: pickle.dump(
    {'pca':pca_8,'scaler':scaler,'n_qubits':8,'embedding_type':'dinov2_768','n_classes':2}, open(p,'wb')))

---
## Step 4: Classical baselines + save

In [ ]:
idx_tr=np.random.RandomState(42).choice(len(X_train_scaled),1000,replace=False)
idx_te=np.random.RandomState(42).choice(len(X_test_scaled),200,replace=False)
X_tr,y_tr=X_train_scaled[idx_tr],train_labels[idx_tr]
X_te,y_te=X_test_scaled[idx_te],test_labels[idx_te]
def bf1(yt,yp): return float(f1_score(yt,yp,average='binary'))
classifiers={'Linear':SVC(kernel='linear',C=1.0,random_state=42),
             'RBF':SVC(kernel='rbf',C=1.0,gamma='scale',random_state=42),
             'Polynomial':SVC(kernel='poly',C=1.0,degree=3,coef0=1,random_state=42)}
results={}
print(f"{'Kernel':<14}{'Acc':>10}{'F1':>10}")
for name,clf in classifiers.items():
    clf.fit(X_tr,y_tr); yp=clf.predict(X_te)
    results[name]={'accuracy':float(accuracy_score(y_te,yp)),'f1':bf1(y_te,yp)}
    print(f"{name:<14}{results[name]['accuracy']:>10.4f}{results[name]['f1']:>10.4f}")
checkpoint('cifar10_dinov2_baseline_results.pkl', lambda p: pickle.dump(results,open(p,'wb')))
def save_final(p):
    np.savez(p, X_train=X_train_scaled,X_test=X_test_scaled,y_train=train_labels,y_test=test_labels,
             X_tr_subset=X_tr,X_te_subset=X_te,y_tr_subset=y_tr,y_te_subset=y_te)
checkpoint('cifar10_dinov2_embeddings.npz', save_final)
loaded=np.load(_find('cifar10_dinov2_embeddings.npz'))
assert loaded['X_tr_subset'].shape==(1000,8); assert loaded['X_te_subset'].shape==(200,8)
print("VERIFIED (1000,8)/(200,8).")
checkpoint('cifar10_dinov2_meta.json', lambda p: json.dump(
    {'dataset':'cifar10_binary','encoder':'dinov2','embedding_type':'dinov2_768',
     'n_classes':2,'metric':'binary','baselines':results}, open(p,'w'),indent=2))
print("\nWeek 2 DINOv2 CIFAR-10 done.")

---
## Done — DINOv2 Week 2

### Saved (tag = `cifar10_dinov2`):
- raw, pca_scaler, baseline_results, embeddings, meta

### For Week 3/4:
Reuse your existing `week3c_cifar10` / `week4c_cifar10` notebooks — change the
loaded filename to `cifar10_dinov2_embeddings.npz`. Binary F1, same quantum kernel.

### The comparison this enables:
CLIP image encoder (vision-language) vs DINOv2 (self-supervised, no language) on
the SAME CIFAR-10 task. If the quantum-vs-classical pattern holds on both, the
finding does not depend on the encoder being language-trained. This is a sharper
generalization claim than another CLIP variant.